# Detección y Clasificación de Enfermedades en Plantas mediante Visión Artificial
### Notebook completo (versión corregida): Data-Set → Entrenamiento → Evaluación → Grad-CAM → Exportación a la app
Curso: Inteligencia Artificial (Principios y Técnicas)

**Cómo ejecutarlo**
1. `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.
2. `Entorno de ejecución → Ejecutar todas`. La primera celda pide permiso para montar Google Drive.
3. Todo lo importante (manifiestos, modelos, métricas, figuras y el modelo para la app) se guarda en
   `MyDrive/proyecto_plantas/`. **Si Colab se reinicia, no se pierde nada**: al volver a ejecutar, las celdas
   reutilizan lo que ya está en Drive (esto es lo que causaba el error `FileNotFoundError: /content/manifest_train.csv`).

**Tiempo aproximado en T4:** preparación ~10 min · fase 1 ~45 min · fase 2 ~25 min · evaluación ~5 min.
La validación cruzada completa (5 folds) es opcional y puede hacerse en varias sesiones (se reanuda sola).

**Cambios respecto a la versión anterior** (resumen; el detalle está al final del notebook):
listado ordenado de archivos (particiones reproducibles), un solo notebook (sin fusión de cuadernos),
guardado en Drive, detección de duplicados más segura, cobertura real de `leaf_id`, preprocesamiento
`preprocess_input` documentado, fase 2 de *fine-tuning*, validación cruzada de 5 folds, evaluación
completa con PlantDoc, Grad-CAM y exportación del modelo para la web app.

In [ ]:
!pip install -q tensorflowjs
!tensorflowjs_converter --input_format=keras modelo_resnet50.h5 model/
!zip -r model.zip model/

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
statsmodels 0.15.0 requires packaging>=21.3, but you have packaging 20.9 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 20.9 which is incompatible.
db-dtypes 1.9.0 requires packaging>=24.2.0, but you have packaging 20.9 which is incompatible.
pandas-gbq 0.30.0 requires packaging>=22.0.0, but you have packaging 20.9 which is incompatible.
astropy 7.2.3 requires packaging>=22.0.0, but you have packaging 20.9 which is incompatible.
langsmith 0.14.17 requires packaging>=23.2, but you have packaging 20.9 which is incompatible.
bigquery-magics 0.14.0 requires packaging>=24.2.0, but you have packaging 20.9 which is incompatible.
jupyter-server 2.2

## 0 — Configuración general

In [ ]:
# ====== PARÁMETROS DEL EXPERIMENTO (documentados para reproducibilidad, RNF01) ======
SEED = 42
IMG_SIZE = (224, 224)
BATCH = 32
NUM_CLASSES = 14

EPOCAS_FASE1 = 25      # backbone congelado, Adam lr=1e-4
EPOCAS_FASE2 = 10      # fine-tuning del bloque conv5, Adam lr=1e-5
LR_FASE1, LR_FASE2 = 1e-4, 1e-5
PACIENCIA = 5          # EarlyStopping sobre val_loss

EJECUTAR_CV_COMPLETA = False   # True = entrena los 5 folds (se reanuda si Colab se corta)
EPOCAS_CV = 10                  # épocas por fold en la validación cruzada (solo fase 1)

FORZAR_REPROCESAR = False       # True = rehace descarga/filtrado/particiones aunque existan en Drive

In [ ]:
import os, re, glob, json, shutil, random, hashlib
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

random.seed(SEED); np.random.seed(SEED)

# --- Google Drive: aquí se guarda todo lo que no se debe perder ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/proyecto_plantas'
except Exception:
    OUT = '/content/proyecto_plantas'   # fuera de Colab
os.makedirs(OUT, exist_ok=True)
for sub in ['manifiestos', 'modelos', 'resultados', 'figuras', 'modelo_web']:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

def ruta(*p): return os.path.join(OUT, *p)
EXTS = ('.jpg', '.jpeg', '.png')
def es_imagen(f): return f.lower().endswith(EXTS)
def listar(d):  # listado ORDENADO: os.listdir no garantiza orden y eso cambiaba las particiones entre ejecuciones
    return sorted(os.listdir(d))
print('Resultados en:', OUT)

# 3.1.4 — Elaboración del Data-Set (PlantVillage + PlantDoc)

## 3.1.4.1 / 3.1.4.2 / 3.1.4.3 — Descarga desde los repositorios oficiales de GitHub

In [ ]:
PV_ROOT, PD_ROOT = '/content/plantvillage', '/content/plantdoc'
if not os.path.exists(PV_ROOT):
    !git clone -q --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git {PV_ROOT}
if not os.path.exists(PD_ROOT):
    !git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git {PD_ROOT}
!pip -q install imagehash
print('PlantVillage:', os.path.exists(PV_ROOT), '| PlantDoc:', os.path.exists(PD_ROOT))

## 3.1.4.4 — Verificación del volumen real de imágenes descargadas

In [ ]:
def contar_imagenes(root):
    return sum(1 for _, _, fs in os.walk(root) for f in fs if es_imagen(f))

PV_COLOR = os.path.join(PV_ROOT, 'raw', 'color')   # solo la variante RGB original (no grayscale ni segmented)
PD_TRAIN, PD_TEST = os.path.join(PD_ROOT, 'train'), os.path.join(PD_ROOT, 'test')
print('Total de imágenes — PlantVillage (color):', contar_imagenes(PV_COLOR))
print('Total de imágenes — PlantDoc            :', contar_imagenes(PD_ROOT))

**Interpretación:** se contabilizaron 54 305 imágenes en PlantVillage (carpeta `raw/color`) y 2 579 en PlantDoc.
Las diferencias con las cifras publicadas (54 306 y 2 598) se deben a la versión de los repositorios.

## 3.1.4.5 — Estructura de carpetas y clases disponibles

In [ ]:
pv_classes = listar(PV_COLOR)
pd_classes = sorted(set(listar(PD_TRAIN)) | set(listar(PD_TEST)))
print(f'PlantVillage: {len(pv_classes)} clases | PlantDoc: {len(pd_classes)} clases')
print('\nPlantVillage:', pv_classes)
print('\nPlantDoc:', pd_classes)

**Interpretación:** PlantVillage tiene 38 carpetas de clase con el formato `Especie___estado`
(p. ej. `Tomato___Late_blight`), mientras que PlantDoc tiene 28 carpetas con otra nomenclatura
(p. ej. `Tomato leaf late blight`). Ambos datasets no comparten el mismo esquema de etiquetas, por lo que
se necesita un mapeo manual de clases equivalentes (3.1.4.8).

## 3.1.4.6 — Selección de especies: tomate, papa y maíz

In [ ]:
KEYWORDS = {'tomate': ['tomato'], 'papa': ['potato'], 'maiz': ['corn', 'maize']}
def especie_de(nombre):
    n = nombre.lower()
    for esp, kws in KEYWORDS.items():
        if any(k in n for k in kws): return esp
    return None

pv_sel = {e: [c for c in pv_classes if especie_de(c) == e] for e in KEYWORDS}
pd_sel = {e: [c for c in pd_classes if especie_de(c) == e] for e in KEYWORDS}
for nombre, sel in [('PlantVillage', pv_sel), ('PlantDoc', pd_sel)]:
    print(f'== {nombre}: {sum(len(v) for v in sel.values())} clases ==')
    for e, cs in sel.items(): print(f'  {e} ({len(cs)}):', cs)

**Interpretación:** se seleccionaron 17 clases de PlantVillage (10 de tomate, 3 de papa y 4 de maíz) y
14 de PlantDoc (9 de tomate, 2 de papa y 3 de maíz). El resto de especies queda fuera del alcance del proyecto.

## 3.1.4.7 — Conteo de imágenes por especie y clase

In [ ]:
def n_imgs(d): return sum(1 for f in listar(d) if es_imagen(f)) if os.path.isdir(d) else 0

df_pv = pd.DataFrame([{'fuente': 'PlantVillage', 'especie': e, 'clase': c, 'n_imagenes': n_imgs(os.path.join(PV_COLOR, c))}
                      for e, cs in pv_sel.items() for c in cs])
df_pd = pd.DataFrame([{'fuente': 'PlantDoc', 'especie': e, 'clase': c,
                       'n_imagenes': n_imgs(os.path.join(PD_TRAIN, c)) + n_imgs(os.path.join(PD_TEST, c))}
                      for e, cs in pd_sel.items() for c in cs])
display(df_pv); display(df_pd)

## 3.1.4.8 — Clases compatibles entre PlantVillage y PlantDoc

In [ ]:
MANUAL_MAP = {
    # TOMATE (9)
    'Tomato___Bacterial_spot': 'Tomato leaf bacterial spot',
    'Tomato___Early_blight': 'Tomato Early blight leaf',
    'Tomato___Late_blight': 'Tomato leaf late blight',
    'Tomato___Leaf_Mold': 'Tomato mold leaf',
    'Tomato___Septoria_leaf_spot': 'Tomato Septoria leaf spot',
    'Tomato___Spider_mites Two-spotted_spider_mite': 'Tomato two spotted spider mites leaf',
    'Tomato___Tomato_mosaic_virus': 'Tomato leaf mosaic virus',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'Tomato leaf yellow virus',
    'Tomato___healthy': 'Tomato leaf',                 # en PlantDoc "Tomato leaf" = hoja sana
    # PAPA (2)
    'Potato___Early_blight': 'Potato leaf early blight',
    'Potato___Late_blight': 'Potato leaf late blight',
    # MAÍZ (3)
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Corn Gray leaf spot',
    'Corn_(maize)___Common_rust_': 'Corn rust leaf',
    'Corn_(maize)___Northern_Leaf_Blight': 'Corn leaf blight',
}
# Sin contraparte en PlantDoc (se excluyen): Tomato___Target_Spot, Potato___healthy, Corn_(maize)___healthy

def normalizar_estado(nombre):
    n = re.sub(r'(tomato|potato|corn|maize)[_\s]*', '', nombre.lower())
    return re.sub(r'[^a-z]+', '_', n).strip('_')

compat = pd.DataFrame([{'especie': especie_de(pv), 'clase_plantvillage': pv, 'clase_plantdoc': pdc}
                       for pv, pdc in MANUAL_MAP.items()])
compat['estado'] = compat['especie'] + '_' + compat['clase_plantvillage'].apply(normalizar_estado)
compat = compat.merge(df_pv[['clase', 'n_imagenes']].rename(columns={'clase': 'clase_plantvillage', 'n_imagenes': 'n_plantvillage'}))
compat = compat.merge(df_pd[['clase', 'n_imagenes']].rename(columns={'clase': 'clase_plantdoc', 'n_imagenes': 'n_plantdoc'}))
compat = compat.sort_values('estado').reset_index(drop=True)

CLASES = compat['estado'].tolist()                       # orden alfabético = índice de la etiqueta
label_to_index = {c: i for i, c in enumerate(CLASES)}
compat['label'] = compat['estado'].map(label_to_index)
assert len(compat) == NUM_CLASSES, f'Se esperaban 14 clases compatibles, hay {len(compat)}'
display(compat)
print('PlantVillage:', compat.n_plantvillage.sum(), '| PlantDoc:', compat.n_plantdoc.sum())

**Interpretación:** se identificaron 14 clases compatibles. Quedan fuera `Tomato___Target_Spot`,
`Potato___healthy` y `Corn_(maize)___healthy`, porque PlantDoc no tiene clases equivalentes.
`Tomato two spotted spider mites leaf` solo tiene 2 imágenes en PlantDoc, por lo que sus métricas
externas no son confiables.

## 3.1.4.9 — Organización del Data-Set final

In [ ]:
FINAL = '/content/dataset_final'
if FORZAR_REPROCESAR and os.path.exists(FINAL): shutil.rmtree(FINAL)
if not os.path.exists(FINAL):
    for _, r in compat.iterrows():
        shutil.copytree(os.path.join(PV_COLOR, r.clase_plantvillage),
                        os.path.join(FINAL, 'plantvillage', r.especie, r.clase_plantvillage))
        for split, d in [('train', PD_TRAIN), ('test', PD_TEST)]:
            src = os.path.join(d, r.clase_plantdoc)
            if os.path.isdir(src):
                shutil.copytree(src, os.path.join(FINAL, 'plantdoc', r.especie, r.clase_plantdoc, split))
print('Estructura:', FINAL, '→ plantvillage/<especie>/<clase> y plantdoc/<especie>/<clase>/<train|test>')

## 3.1.4.10 — Cantidades finales del Data-Set

In [ ]:
resumen = pd.concat([
    compat.assign(fuente='PlantVillage', clase=compat.clase_plantvillage, n_imagenes=compat.n_plantvillage),
    compat.assign(fuente='PlantDoc', clase=compat.clase_plantdoc, n_imagenes=compat.n_plantdoc),
])[['fuente', 'especie', 'clase', 'estado', 'n_imagenes']]
display(resumen.groupby(['fuente', 'especie'])['n_imagenes'].sum().unstack(0))
resumen.to_csv(ruta('resultados', 'resumen_dataset_compatible.csv'), index=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, fuente in zip(axes, ['PlantVillage', 'PlantDoc']):
    d = resumen[resumen.fuente == fuente].sort_values('n_imagenes', ascending=False)
    ax.bar(d['estado'], d['n_imagenes']); ax.set_title(f'Imágenes por clase — {fuente}')
    ax.set_ylabel('Número de imágenes'); ax.tick_params(axis='x', rotation=75)
plt.tight_layout(); plt.savefig(ruta('figuras', 'distribucion_14_clases.png'), dpi=200, bbox_inches='tight'); plt.show()

**Interpretación:** antes del filtrado, el Data-Set compatible tiene 21 446 imágenes de PlantVillage
(tomate 16 756, papa 2 000, maíz 2 690) y 1 344 de PlantDoc (tomate 746, papa 222, maíz 376).
Hay un desbalance fuerte entre clases (por ejemplo, 5 357 imágenes de *Yellow Leaf Curl Virus* frente a 373
de *mosaic virus*), que se compensa con **pesos por clase** durante el entrenamiento (4.2). El aumento de datos
no corrige el desbalance por sí solo: su función es dar variabilidad visual.

# 3.2 — Normalización y/o Filtrado de Datos

### 3.2.1 — Filtrado de imágenes corruptas (PlantVillage y PlantDoc)

In [ ]:
def imagenes_corruptas(root):
    malas = []
    for dp, _, fs in sorted(os.walk(root)):
        for f in sorted(fs):
            if not es_imagen(f): continue
            p = os.path.join(dp, f)
            try:
                with Image.open(p) as im: im.verify()
                with Image.open(p) as im: im.convert('RGB').load()
            except Exception:
                malas.append(p)
    return malas

PV_FINAL, PD_FINAL = os.path.join(FINAL, 'plantvillage'), os.path.join(FINAL, 'plantdoc')
for nombre, d in [('PlantVillage', PV_FINAL), ('PlantDoc', PD_FINAL)]:
    malas = imagenes_corruptas(d)
    for p in malas: os.remove(p)
    print(f'{nombre}: {len(malas)} imágenes corruptas eliminadas')

### 3.2.2 — Filtrado de imágenes duplicadas
Se combinan dos criterios: (1) duplicados **exactos** por hash MD5 del archivo y (2) duplicados **visuales**
por hash perceptual (pHash, distancia 0) **dentro de la misma clase**. La versión anterior usaba `average_hash`
sobre todas las clases a la vez; ese hash de 64 bits colisiona con facilidad en PlantVillage (fondos uniformes),
por lo que podía eliminar imágenes distintas, y el resultado dependía del orden de lectura de los archivos.

In [ ]:
import imagehash
def duplicados(root):
    eliminar, por_clase = [], {}
    vistos_md5 = {}
    for dp, _, fs in sorted(os.walk(root)):
        for f in sorted(fs):
            if not es_imagen(f): continue
            p = os.path.join(dp, f)
            md5 = hashlib.md5(open(p, 'rb').read()).hexdigest()
            if md5 in vistos_md5: eliminar.append(p); continue
            vistos_md5[md5] = p
            with Image.open(p) as im: h = str(imagehash.phash(im.convert('RGB')))
            vistos = por_clase.setdefault(dp, set())
            if h in vistos: eliminar.append(p)
            else: vistos.add(h)
    return eliminar

dups = duplicados(PV_FINAL)
for p in dups: os.remove(p)
print('Imágenes duplicadas eliminadas en PlantVillage:', len(dups))

### 3.2.3 — Redimensionamiento, conversión a RGB y normalización de píxeles
Cada imagen se convierte a RGB y se redimensiona a 224×224. Como el modelo usa **ResNet50 preentrenada en ImageNet**,
la normalización debe ser la misma que se usó al preentrenarla: `tf.keras.applications.resnet50.preprocess_input`
(convierte RGB→BGR y resta la media de ImageNet por canal; el rango resultante es aproximadamente −124 a 151).
Normalizar a [0, 1] con ResNet50 de Keras sería un error, porque las capas preentrenadas esperan esa otra escala.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.resnet50 import preprocess_input
tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

def cargar_imagen(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)  # RGB, uint8
    return tf.image.resize(img, IMG_SIZE)                                                   # float32, 0-255

ejemplo = sorted(glob.glob(os.path.join(PV_FINAL, '*', '*', '*')))[0]
x = cargar_imagen(ejemplo)
print('Forma:', x.shape, '| rango original:', float(tf.reduce_min(x)), '-', float(tf.reduce_max(x)))
xp = preprocess_input(x[None])
print('Rango tras preprocess_input:', round(float(tf.reduce_min(xp)), 2), 'a', round(float(tf.reduce_max(xp)), 2))

### 3.2.4 — Codificación de etiquetas

In [ ]:
for e, i in label_to_index.items(): print(f'{i:2d}: {e}')
json.dump(CLASES, open(ruta('resultados', 'clases.json'), 'w'), ensure_ascii=False, indent=1)

### 3.2.5 — Aumento de datos (data augmentation)
Se aplica solo al entrenamiento: volteo horizontal, rotación de ±20° (0,0556 de vuelta) y brillo ±20 %.

In [ ]:
augment = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(20 / 360),
    keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
], name='augmentation')

# 3.3 — Organización del Data-Set

### 3.3.0 — Manifiesto de archivos y etiquetas

In [ ]:
def manifiesto_pv():
    filas = []
    for _, r in compat.iterrows():
        d = os.path.join(PV_FINAL, r.especie, r.clase_plantvillage)
        for f in listar(d):
            if es_imagen(f):
                filas.append({'filepath': os.path.join(d, f), 'filename': f, 'especie': r.especie,
                              'clase': r.clase_plantvillage, 'estado': r.estado, 'label': r.label})
    return pd.DataFrame(filas)

manifest = manifiesto_pv()
print('Imágenes:', len(manifest), '| clases:', manifest.label.nunique())
display(manifest.groupby(['label', 'estado']).size().rename('n_imagenes').reset_index())

### 3.3.0b — Identificador de hoja (leaf_id) para evitar fuga de información
PlantVillage incluye `leaf_grouping/leaf-map.json`, que agrupa las fotos tomadas a la **misma hoja**.
Si dos fotos de la misma hoja quedan una en entrenamiento y otra en prueba, la prueba se vuelve optimista.
Se corrige la búsqueda para elegir el grupo de la clase correcta y se informa la **cobertura real**:
el archivo no contiene entradas para maíz ni para parte de tomate, así que esas imágenes quedan con un
identificador propio (`img_...`). Para ellas no es posible garantizar la separación por hoja.

In [ ]:
leaf_map = json.load(open(os.path.join(PV_ROOT, 'leaf_grouping', 'leaf-map.json'), encoding='utf-8'))

def leaf_id(fila):
    s = fila.filename.replace('_final_masked', '')
    if '___' in s: s = s.split('___')[-1]
    s = re.sub(r'\.(jpg|jpeg|png)$', '', s, flags=re.I)
    s = re.sub(r'\s*copy.*$', '', s, flags=re.I).strip().lower()
    for sug in leaf_map.get(s, []):
        if sug.split(':::')[0] == fila.clase:
            return sug
    return 'img_' + fila.filename

manifest['leaf_id'] = manifest.apply(leaf_id, axis=1)
manifest['con_leaf_map'] = ~manifest.leaf_id.str.startswith('img_')
cob = manifest.groupby('estado').con_leaf_map.mean().mul(100).round(1).rename('% con leaf_id real')
display(cob.to_frame())
print(f'Cobertura total: {manifest.con_leaf_map.mean():.1%} | leaf_id únicos: {manifest.leaf_id.nunique()}')

### 3.3.1 / 3.3.2 — División entrenamiento (≈80 %) / prueba (≈20 %) estratificada y agrupada por hoja

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

f_train, f_test = ruta('manifiestos', 'manifest_train.csv'), ruta('manifiestos', 'manifest_test.csv')
if os.path.exists(f_train) and not FORZAR_REPROCESAR:
    train_df, test_df = pd.read_csv(f_train), pd.read_csv(f_test)
    print('Particiones cargadas desde Drive (no se recalculan).')
else:
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    tr, te = next(sgkf.split(manifest.filepath, manifest.label, manifest.leaf_id))
    train_df, test_df = manifest.iloc[tr].reset_index(drop=True), manifest.iloc[te].reset_index(drop=True)
    train_df.to_csv(f_train, index=False); test_df.to_csv(f_test, index=False)

total = len(train_df) + len(test_df)
print(f'ENTRENAMIENTO: {len(train_df)} ({len(train_df)/total:.2%}) | PRUEBA: {len(test_df)} ({len(test_df)/total:.2%})')
assert not set(train_df.leaf_id) & set(test_df.leaf_id)
print('Leaf IDs compartidos entre train y test: 0')

### 3.3.3 — Validación cruzada estratificada y agrupada (5 folds) sobre el entrenamiento
Se usa `StratifiedGroupKFold` (la celda anterior con `StratifiedKFold` se eliminó porque no respetaba el `leaf_id`).

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
FOLDS = list(cv.split(train_df.filepath, train_df.label, train_df.leaf_id))
info = []
for k, (tr, va) in enumerate(FOLDS, 1):
    assert not set(train_df.leaf_id.iloc[tr]) & set(train_df.leaf_id.iloc[va])
    info.append({'fold': k, 'entrenamiento': len(tr), 'validacion': len(va), 'leaf_ids_compartidos': 0})
fold_info = pd.DataFrame(info); display(fold_info)
fold_info.to_csv(ruta('manifiestos', 'cross_validation_folds.csv'), index=False)

### 3.3.4 — Conjunto de evaluación externa (PlantDoc)
Las imágenes de PlantDoc (train + test del repositorio) se usan **solo** para la evaluación externa (RNF04).
Se convierten a JPEG RGB para evitar formatos que TensorFlow no decodifica (CMYK, PNG con transparencia, etc.).

In [ ]:
PD_RGB = '/content/plantdoc_rgb'
f_pd = ruta('manifiestos', 'manifest_plantdoc.csv')
filas = []
for _, r in compat.iterrows():
    for split in ['train', 'test']:
        d = os.path.join(PD_FINAL, r.especie, r.clase_plantdoc, split)
        if not os.path.isdir(d): continue
        for f in listar(d):
            if not es_imagen(f): continue
            dst = os.path.join(PD_RGB, r.estado, f'{split}_{os.path.splitext(f)[0]}.jpg')
            if not os.path.exists(dst):
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with Image.open(os.path.join(d, f)) as im: im.convert('RGB').save(dst, quality=95)
            filas.append({'filepath': dst, 'clase': r.clase_plantdoc, 'estado': r.estado, 'label': r.label, 'split_original': split})
plantdoc_df = pd.DataFrame(filas); plantdoc_df.to_csv(f_pd, index=False)
print('Imágenes PlantDoc para evaluación externa:', len(plantdoc_df))
display(plantdoc_df.groupby('estado').size().rename('n').to_frame())

# 4 — Aprendizaje: ResNet50 con Transfer Learning

### 4.2.1 — Canal de datos (tf.data) y modelo

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
def _leer(path, label): return cargar_imagen(path), label

def make_ds(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df.filepath.values, df.label.values.astype('int32')))
    if training: ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_leer, num_parallel_calls=AUTOTUNE).batch(BATCH)
    if training: ds = ds.map(lambda x, y: (preprocess_input(augment(x, training=True)), y), num_parallel_calls=AUTOTUNE)
    else:        ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

def crear_modelo():
    base = keras.applications.ResNet50(include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inp = keras.Input(IMG_SIZE + (3,))
    x = base(inp, training=False)          # BatchNorm siempre en modo inferencia (también en fine-tuning)
    x = keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = keras.layers.Dropout(0.3, name='dropout')(x)
    out = keras.layers.Dense(NUM_CLASSES, activation='softmax', name='clasificador')(x)
    return keras.Model(inp, out, name='resnet50_plantas'), base

def pesos_clase(df):
    from sklearn.utils.class_weight import compute_class_weight
    w = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=df.label.values)
    return dict(enumerate(w))

modelo_demo, _ = crear_modelo(); modelo_demo.summary(); del modelo_demo

### 4.2.2 — Entrenamiento en dos fases
- **Fase 1:** backbone congelado, solo se entrena la cabeza (GAP → Dropout 0,3 → Dense 14 softmax). Adam lr = 1e-4.
- **Fase 2 (fine-tuning):** se descongela el bloque `conv5` de ResNet50 y se entrena con lr = 1e-5.

La validación usa el **fold 1** (partición agrupada por hoja) para EarlyStopping y para elegir el mejor checkpoint.
Si Colab se reinicia, los modelos ya guardados en Drive no se vuelven a entrenar.

In [ ]:
tr, va = FOLDS[0]
fit_df, val_df = train_df.iloc[tr].reset_index(drop=True), train_df.iloc[va].reset_index(drop=True)
train_ds, val_ds = make_ds(fit_df, True), make_ds(val_df, False)
CW = pesos_clase(fit_df)
print('Entrenamiento:', len(fit_df), '| Validación:', len(val_df))
print('Pesos por clase:', {k: round(v, 2) for k, v in CW.items()})

M1, M2 = ruta('modelos', 'resnet50_fase1.keras'), ruta('modelos', 'resnet50_final.keras')
def callbacks(path):
    return [keras.callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(path, monitor='val_loss', save_best_only=True),
            keras.callbacks.CSVLogger(path.replace('.keras', '_historial.csv'))]

if os.path.exists(M1):
    model = keras.models.load_model(M1); print('Fase 1 cargada desde Drive.')
else:
    model, base = crear_modelo()
    model.compile(optimizer=keras.optimizers.Adam(LR_FASE1), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCAS_FASE1, class_weight=CW, callbacks=callbacks(M1))
    model.save(M1)

In [ ]:
# ---- Fase 2: fine-tuning de conv5 ----
if os.path.exists(M2):
    model = keras.models.load_model(M2); print('Modelo final cargado desde Drive.')
else:
    base = model.get_layer('resnet50')
    base.trainable = True
    for layer in base.layers:
        layer.trainable = layer.name.startswith('conv5_') and not isinstance(layer, keras.layers.BatchNormalization)
    model.compile(optimizer=keras.optimizers.Adam(LR_FASE2), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print('Parámetros entrenables:', sum(np.prod(w.shape) for w in model.trainable_weights))
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCAS_FASE2, class_weight=CW, callbacks=callbacks(M2))
    model.save(M2)

### 4.2.3 — Curvas de aprendizaje

In [ ]:
h1 = pd.read_csv(M1.replace('.keras', '_historial.csv')); h1['fase'] = 1
h2p = M2.replace('.keras', '_historial.csv')
h = pd.concat([h1, pd.read_csv(h2p).assign(fase=2)] if os.path.exists(h2p) else [h1], ignore_index=True)
h['epoca_total'] = range(1, len(h) + 1)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, m, t in zip(ax, ['loss', 'accuracy'], ['Pérdida', 'Accuracy']):
    a.plot(h.epoca_total, h[m], label='entrenamiento'); a.plot(h.epoca_total, h['val_' + m], label='validación')
    if (h.fase == 2).any(): a.axvline(len(h1) + 0.5, ls='--', c='gray', label='inicio fine-tuning')
    a.set_title(t); a.set_xlabel('Época'); a.legend()
plt.tight_layout(); plt.savefig(ruta('figuras', 'curvas_aprendizaje.png'), dpi=200, bbox_inches='tight'); plt.show()
print('Mejor val_loss fase 1:', round(h1.val_loss.min(), 4), 'en época', int(h1.val_loss.idxmin()) + 1)

### 4.2.4 — Validación cruzada de 5 folds (opcional)
Con `EJECUTAR_CV_COMPLETA = True` se entrena un modelo por fold (solo fase 1, `EPOCAS_CV` épocas) y se guarda
cada resultado en Drive; si la sesión se corta, al volver a ejecutar continúa desde el fold pendiente.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
f_cv = ruta('resultados', 'cv_5folds.csv')
cv_res = pd.read_csv(f_cv) if os.path.exists(f_cv) else pd.DataFrame()
if EJECUTAR_CV_COMPLETA:
    for k, (tr, va) in enumerate(FOLDS, 1):
        if len(cv_res) and k in cv_res.fold.values: continue
        a, b = train_df.iloc[tr], train_df.iloc[va]
        m, _ = crear_modelo()
        m.compile(optimizer=keras.optimizers.Adam(LR_FASE1), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        m.fit(make_ds(a, True), validation_data=make_ds(b), epochs=EPOCAS_CV, class_weight=pesos_clase(a),
              callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, restore_best_weights=True)], verbose=2)
        yp = m.predict(make_ds(b), verbose=0).argmax(1)
        p, r, f, _ = precision_recall_fscore_support(b.label, yp, average='macro', zero_division=0)
        cv_res = pd.concat([cv_res, pd.DataFrame([{'fold': k, 'accuracy': accuracy_score(b.label, yp),
                                                   'precision_macro': p, 'recall_macro': r, 'f1_macro': f}])], ignore_index=True)
        cv_res.to_csv(f_cv, index=False); keras.backend.clear_session()
if len(cv_res):
    display(cv_res.round(4))
    print('Media ± desviación estándar:'); display(cv_res.drop(columns='fold').agg(['mean', 'std']).round(4))
else:
    print('Validación cruzada no ejecutada (EJECUTAR_CV_COMPLETA = False).')

# 5 — Comprobación: conjunto de entrenamiento y de prueba (PlantVillage)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
if 'model' not in globals(): model = keras.models.load_model(M2 if os.path.exists(M2) else M1)

def evaluar(df, nombre):
    probs = model.predict(make_ds(df), verbose=0)
    y, yp = df.label.values, probs.argmax(1)
    presentes = sorted(set(y))                   # macro solo sobre clases presentes en el conjunto
    p, r, f, _ = precision_recall_fscore_support(y, yp, labels=presentes, average='macro', zero_division=0)
    res = {'conjunto': nombre, 'n': len(df), 'accuracy': accuracy_score(y, yp), 'precision_macro': p,
           'recall_macro': r, 'f1_macro': f, 'f1_ponderado': f1_score(y, yp, average='weighted', zero_division=0),
           'confianza_media': probs.max(1).mean()}
    return res, y, yp, probs

def matriz(y, yp, titulo, archivo):
    fig, ax = plt.subplots(figsize=(11, 9))
    ConfusionMatrixDisplay(confusion_matrix(y, yp, labels=range(NUM_CLASSES), normalize='true'),
                           display_labels=CLASES).plot(ax=ax, cmap='Greens', values_format='.2f', colorbar=False)
    ax.set_title(titulo); plt.xticks(rotation=75); plt.tight_layout()
    plt.savefig(ruta('figuras', archivo), dpi=200, bbox_inches='tight'); plt.show()

muestra_train = train_df.sample(3000, random_state=SEED)       # sin aumento de datos
r_train, *_ = evaluar(muestra_train, 'Entrenamiento (muestra 3 000)')
r_test, y_te, yp_te, pr_te = evaluar(test_df, 'Prueba PlantVillage')
tabla = pd.DataFrame([r_train, r_test]).set_index('conjunto').round(4); display(tabla)
print(f"Brecha de accuracy entrenamiento - prueba: {100*(r_train['accuracy']-r_test['accuracy']):.2f} puntos")
print(classification_report(y_te, yp_te, labels=range(NUM_CLASSES), target_names=CLASES, digits=4, zero_division=0))
matriz(y_te, yp_te, 'Matriz de confusión normalizada — Prueba PlantVillage', 'matriz_confusion_plantvillage.png')

# 6 — Evaluación externa con PlantDoc (cambio de dominio)

In [ ]:
r_pd, y_pd, yp_pd, pr_pd = evaluar(plantdoc_df, 'PlantDoc (externa)')
comp = pd.DataFrame([r_test, r_pd]).set_index('conjunto')
comp.loc['Diferencia'] = comp.iloc[1, 1:] - comp.iloc[0, 1:]
display(comp.round(4))
presentes = sorted(set(y_pd))
print(classification_report(y_pd, yp_pd, labels=presentes, target_names=[CLASES[i] for i in presentes], digits=4, zero_division=0))
matriz(y_pd, yp_pd, 'Matriz de confusión normalizada — PlantDoc', 'matriz_confusion_plantdoc.png')

pred_cnt = pd.Series(yp_pd).map(dict(enumerate(CLASES))).value_counts()
print('Clases más predichas en PlantDoc (sesgo hacia una clase = síntoma de cambio de dominio):'); display(pred_cnt.head(5))
pd.concat([tabla.reset_index(), comp.reset_index()]).to_csv(ruta('resultados', 'metricas.csv'), index=False)

**Confianza del modelo:** la columna `confianza_media` compara la probabilidad máxima promedio en PlantVillage
y en PlantDoc. Si el modelo sigue muy seguro en PlantDoc aunque acierte mucho menos, está **sobreconfiado** fuera
de su dominio (Xiang, Shi y Zhu, 2026). Por eso la app muestra una advertencia cuando la confianza es baja y no
presenta la predicción como diagnóstico definitivo.

# 6.x — Interpretabilidad con Grad-CAM

In [ ]:
import cv2
base = model.get_layer('resnet50')
gap, dense = model.get_layer('gap'), model.get_layer('clasificador')

def grad_cam(path):
    img = cargar_imagen(path)
    x = preprocess_input(img[None])
    with tf.GradientTape() as tape:
        A = base(x, training=False)                 # mapas de activación conv5_block3_out (7x7x2048)
        tape.watch(A)
        probs = dense(gap(A))
        c = int(tf.argmax(probs[0])); score = probs[:, c]
    g = tape.gradient(score, A)
    w = tf.reduce_mean(g, axis=(1, 2))              # importancia de cada mapa
    cam = tf.nn.relu(tf.reduce_sum(A[0] * w[0], -1)).numpy()
    cam = cv2.resize(cam / (cam.max() + 1e-8), IMG_SIZE)
    return img.numpy().astype('uint8'), cam, c, float(probs[0, c])

def mostrar_gradcam(df, titulo, archivo, n=6):
    filas = df.groupby('label').head(1).sample(min(n, df.label.nunique()), random_state=SEED)
    fig, ax = plt.subplots(2, len(filas), figsize=(3.2 * len(filas), 6.6))
    for j, (_, r) in enumerate(filas.iterrows()):
        img, cam, c, p = grad_cam(r.filepath)
        ax[0, j].imshow(img); ax[0, j].set_title(f'Real: {r.estado}', fontsize=7)
        ax[1, j].imshow(img); ax[1, j].imshow(cam, cmap='jet', alpha=0.45)
        ax[1, j].set_title(f'Pred: {CLASES[c]} ({p:.0%})', fontsize=7, color='green' if c == r.label else 'red')
        for a in ax[:, j]: a.axis('off')
    fig.suptitle(titulo); plt.tight_layout(); plt.savefig(ruta('figuras', archivo), dpi=200, bbox_inches='tight'); plt.show()

mostrar_gradcam(test_df, 'Grad-CAM — PlantVillage (prueba)', 'gradcam_plantvillage.png')
mostrar_gradcam(plantdoc_df, 'Grad-CAM — PlantDoc (externa)', 'gradcam_plantdoc.png')

**Interpretación:** en PlantVillage se espera que el mapa se concentre en las lesiones de la hoja. En PlantDoc,
si el mapa cae sobre el fondo, otras hojas o bordes de la foto, es evidencia de que el modelo usa señales del
entorno que no se transfieren entre dominios.

# 7 — Exportación del modelo para la web app
Se genera la carpeta `MyDrive/proyecto_plantas/modelo_web/` con `manifest.json` y archivos `pesos_XX.bin`
(≈47 MB en float16). La app ejecuta exactamente la misma ResNet50 en el navegador con TensorFlow.js.
No se necesita `tensorflowjs_converter` (que suele romper la instalación de TensorFlow en Colab).

**Para usarla:** descarguen el zip `modelo_web.zip` y súbanlo a la app (o envíenselo a Claude para dejarlo
integrado en la app y que cargue sola).

In [ ]:
import os, json
import numpy as np
from tensorflow import keras


def exportar_modelo_web(model, class_names, out_dir, shard_mb=18, dtype='float16'):
    """Exporta un modelo ResNet50 + GAP + Dense (Keras) a un formato ligero para la web app.

    - Fusiona cada BatchNormalization con su convolución (W' = W*g/sqrt(v+eps), b' = (b-m)*g/sqrt(v+eps)+beta).
    - Guarda los pesos en float16, divididos en archivos .bin de ~18 MB.
    - Escribe manifest.json con la arquitectura, el preprocesamiento y los nombres de clase.
    No requiere tensorflowjs ni otros convertidores.
    """
    os.makedirs(out_dir, exist_ok=True)
    base = next(l for l in model.layers if isinstance(l, keras.Model))
    dense = [l for l in model.layers if isinstance(l, keras.layers.Dense)][-1]

    tensores = []  # (nombre, array float32)
    for layer in base.layers:
        if not isinstance(layer, keras.layers.Conv2D):
            continue
        nombre = layer.name[:-len('_conv')]           # p. ej. conv2_block1_1
        bn = base.get_layer(nombre + '_bn')
        k, b = [w.astype(np.float32) for w in layer.get_weights()]
        gamma, beta, mean, var = [w.astype(np.float32) for w in bn.get_weights()]
        escala = gamma / np.sqrt(var + bn.epsilon)
        tensores.append((nombre + '/w', k * escala))
        tensores.append((nombre + '/b', (b - mean) * escala + beta))
    kd, bd = dense.get_weights()
    tensores.append(('dense/w', kd.astype(np.float32)))
    tensores.append(('dense/b', bd.astype(np.float32)))

    npdtype = np.float16 if dtype == 'float16' else np.float32
    limite = shard_mb * 1024 * 1024
    shards, actual, entradas, offset_global = [], bytearray(), [], 0
    for nombre, arr in tensores:
        datos = arr.astype(npdtype).tobytes()
        entradas.append({'name': nombre, 'shape': list(arr.shape), 'offset': offset_global, 'bytes': len(datos)})
        offset_global += len(datos)
        actual += datos
        while len(actual) >= limite:
            shards.append(bytes(actual[:limite]))
            actual = actual[limite:]
    if actual:
        shards.append(bytes(actual))

    archivos = []
    for i, s in enumerate(shards):
        fn = f'pesos_{i + 1:02d}.bin'
        with open(os.path.join(out_dir, fn), 'wb') as f:
            f.write(s)
        archivos.append({'file': fn, 'bytes': len(s)})

    manifest = {
        'format': 'resnet50-bnfold-v1',
        'dtype': dtype,
        'input_size': 224,
        'preprocess': 'caffe',  # igual que keras.applications.resnet50.preprocess_input
        'classes': list(class_names),
        'tensors': entradas,
        'shards': archivos,
        'total_bytes': offset_global,
    }
    with open(os.path.join(out_dir, 'manifest.json'), 'w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False)
    print(f'Exportado en {out_dir}: {len(archivos)} archivos .bin, {offset_global / 1e6:.1f} MB')
    return manifest

In [ ]:
WEB = ruta('modelo_web')
exportar_modelo_web(model, CLASES, WEB)
shutil.make_archive(ruta('modelo_web'), 'zip', WEB)
print('Zip listo:', ruta('modelo_web.zip'))

# Verificación: la versión exportada (BN fusionado, float16) da la misma predicción que Keras
man = json.load(open(os.path.join(WEB, 'manifest.json')))
blob = b''.join(open(os.path.join(WEB, s['file']), 'rb').read() for s in man['shards'])
Wt = {t['name']: np.frombuffer(blob[t['offset']:t['offset'] + t['bytes']], dtype=np.float16).astype(np.float32).reshape(t['shape'])
      for t in man['tensors']}
x = preprocess_input(cargar_imagen(test_df.filepath[0])[None]).numpy()
def conv(x, n, s, pad):
    return tf.nn.conv2d(x, Wt[n + '/w'], s, pad.upper()) + Wt[n + '/b']
def bloque(x, n, s, atajo):
    sc = conv(x, n + '_0', s, 'valid') if atajo else x
    y = tf.nn.relu(conv(x, n + '_1', s, 'valid')); y = tf.nn.relu(conv(y, n + '_2', 1, 'same'))
    return tf.nn.relu(sc + conv(y, n + '_3', 1, 'valid'))
y = tf.nn.relu(conv(tf.pad(x, [[0, 0], [3, 3], [3, 3], [0, 0]]), 'conv1', 2, 'valid'))
y = tf.nn.max_pool2d(tf.pad(y, [[0, 0], [1, 1], [1, 1], [0, 0]]), 3, 2, 'VALID')
for n, nb, s in [('conv2', 3, 1), ('conv3', 4, 2), ('conv4', 6, 2), ('conv5', 3, 2)]:
    y = bloque(y, n + '_block1', s, True)
    for i in range(2, nb + 1): y = bloque(y, f'{n}_block{i}', 1, False)
p_web = tf.nn.softmax(tf.reduce_mean(y, (1, 2)) @ Wt['dense/w'] + Wt['dense/b']).numpy()[0]
p_keras = model.predict(x, verbose=0)[0]
print('Diferencia máxima de probabilidad Keras vs. exportado:', float(np.abs(p_web - p_keras).max()))
assert p_web.argmax() == p_keras.argmax()

In [ ]:
try:
    from google.colab import files
    files.download(ruta('modelo_web.zip'))
except Exception as e:
    print('Descárgalo manualmente desde Drive:', ruta('modelo_web.zip'))

# Registro de correcciones respecto al notebook anterior
| # | Problema | Corrección |
|---|---|---|
| 1 | `FileNotFoundError: /content/manifest_train.csv`: al reiniciarse Colab se borra `/content` | Todo se guarda en Google Drive y se reutiliza |
| 2 | Las particiones cambiaban entre ejecuciones (17 073/4 273, 17 081/4 265, 17 049/4 297) porque `os.listdir`/`glob` no tienen orden fijo | Listados ordenados con `sorted()` → resultados reproducibles (RNF01) |
| 3 | Fusión de dos cuadernos por JSON (celda que solo verificaba un archivo) | Un solo notebook de principio a fin |
| 4 | Duplicados con `average_hash` sobre todas las clases (colisiones, dependiente del orden) | MD5 exacto + pHash dentro de cada clase |
| 5 | Filtrado de corruptas solo en PlantVillage | También en PlantDoc, y conversión de PlantDoc a JPEG RGB |
| 6 | El informe decía normalización [0, 1], pero el entrenamiento usa `preprocess_input` | Documentado: se usa `preprocess_input` (escala de ImageNet) |
| 7 | `leaf-map.json` no cubre maíz ni parte de tomate (~39 % de imágenes sin leaf_id real) y la búsqueda no filtraba por clase | Búsqueda por clase + reporte de cobertura |
| 8 | Celda 3.3.2 con `StratifiedKFold` (ignoraba el leaf_id) | Eliminada; solo `StratifiedGroupKFold` |
| 9 | Conteo de PlantDoc contaba cualquier archivo | Solo imágenes |
| 10 | Interpretaciones con `[N]` sin completar | Completadas |
| 11 | Faltaban fine-tuning, CV de 5 folds, evaluación PlantDoc completa y Grad-CAM | Agregados |
| 12 | Aumento de datos presentado como solución al desbalance | Se usan pesos por clase para el desbalance; el aumento aporta variabilidad |